# VN Address Intelligence - Ablation Study (N=5000)

Chạy 4 cấu hình Ablation Study trên Google Colab với GPU T4.

## Cấu hình

- **A1:** NER + TF-IDF + LLM (Full)
- **A2:** NER + TF-IDF (No LLM)
- **A3:** TF-IDF only (Retrieval-only)
- **A4:** NER + LLM (No Retrieval)

**Lưu ý:** Sử dụng TF-IDF retriever thay vì mGTE do vấn đề tương thích CUDA với dữ liệu tiếng Việt.

## Thời gian ước tính

- Cohort size: **N=5000** (x5 so với baseline)
- Mỗi cấu hình: ~1.5-2.5 giờ
- Tổng: ~8-10 giờ
- Total output: **20,000 rows** (4 configs x 5000)

In [ ]:
# === CELL 1: Setup & Mount Drive ===
from google.colab import drive
import sys
import zipfile

# Mount Google Drive
drive.mount('/content/drive')

# Install dependencies
print("Installing dependencies...")
!pip install -q torch transformers sentence-transformers pandas tqdm seqeval pyvi scikit-learn

# Copy files from Drive
print("\nCopying files from Drive...")
!cp /content/drive/MyDrive/colab_vnai/ground_truth.db /content/
!cp /content/drive/MyDrive/colab_vnai/noise_profiles.json /content/
!cp /content/drive/MyDrive/colab_vnai/vnai_src.zip /content/

# Extract source code
print("\nExtracting source code...")
with zipfile.ZipFile('/content/vnai_src.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')

# Add to Python path
sys.path.insert(0, '/content/src')

print("\n[OK] Setup completed")
print(f"Python path: {sys.path[0]}")

In [ ]:
# === CELL 2: Load Models (GPU) ===
import torch
import sqlite3
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from app.ai.models.ner_model import AddressNER
from app.ai.models.llm_model import LLMQwen3

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Load NER
print("\nLoading NER model...")
ner = AddressNER(device="cuda")
print("[OK] NER loaded")

# Create TF-IDF retriever (fallback for mGTE compatibility issues)
print("\n[INFO] Using TF-IDF retriever (mGTE has CUDA compatibility issues)")
print("Loading corpus from ground_truth.db...")

conn = sqlite3.connect("/content/ground_truth.db")
cursor = conn.cursor()
cursor.execute("SELECT address FROM ground_truth LIMIT 10000")
corpus_addresses = [row[0] for row in cursor.fetchall() if row[0]]
conn.close()

print(f"Loaded {len(corpus_addresses)} addresses")

class SimpleTFIDFRetriever:
    """Simple TF-IDF based retriever."""
    def __init__(self, corpus):
        self.corpus = corpus
        self.vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 3))
        print("Encoding corpus with TF-IDF...")
        self.corpus_vectors = self.vectorizer.fit_transform(corpus)
        print(f"[OK] Corpus encoded: {self.corpus_vectors.shape}")
    
    def retrieve_top_k(self, query, top_k=5):
        query_vec = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vec, self.corpus_vectors)[0]
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        return [self.corpus[i] for i in top_indices]

retriever = SimpleTFIDFRetriever(corpus_addresses)
print("[OK] TF-IDF retriever ready")

# Load LLM
print("\nLoading LLM (Qwen 4-bit)...")
llm = LLMQwen3(use_quantization=True, quantization_bits=4, device="cuda")
print("[OK] LLM loaded")

print("\n" + "="*60)
print("All models loaded successfully!")
print("Using TF-IDF retriever instead of mGTE")
print("="*60)

In [ ]:
# === CELL 3: Extract Cohort & Apply Noise ===
import sqlite3
import json
import random
import re
import unicodedata
from typing import Dict, Any

# Load noise profiles
with open("/content/noise_profiles.json", "r", encoding="utf-8") as f:
    NOISE_PROFILES = json.load(f)

def apply_noise(address: str, profile_name: str = "SUP-1.0.0") -> str:
    """Apply noise theo profile (simplified from supa_benchmark.py)."""
    profile = NOISE_PROFILES[profile_name]
    params = profile.get("parameters", {})
    s = address
    
    # Add prefix
    if random.random() < params.get("prefix_prob", 0):
        prefixes = profile.get("prefixes", [""])
        s = random.choice(prefixes) + s
    
    # Add suffix
    if random.random() < params.get("suffix_prob", 0):
        suffixes = profile.get("suffixes", [""])
        s = s + random.choice(suffixes)
    
    # Abbreviate admin units
    if random.random() < params.get("abbreviate_prob", 0):
        s = re.sub(r"(?i)\bThành phố\b", random.choice(["TP.", "TP", "tp"]), s)
        s = re.sub(r"(?i)\bQuận\b", random.choice(["Q.", "Q", "q"]), s)
        s = re.sub(r"(?i)\bPhường\b", random.choice(["P.", "P", "p"]), s)
        s = re.sub(r"(?i)\bĐường\b", random.choice(["Đ.", "đ.", "D."]), s)
    
    # Slang/regional variants
    if random.random() < params.get("slang_prob", 0):
        s = re.sub(r"(?i)\bNgõ\b", random.choice(["hẻm", "ngo"]), s)
        s = re.sub(r"(?i)\bHẻm\b", random.choice(["ngõ", "hem"]), s)
    
    # IME errors (typos)
    if random.random() < params.get("ime_error_prob", 0):
        s = s.replace("đ", random.choice(["dđ", "đd", "d"]))
        s = s.replace("Đ", random.choice(["DĐ", "ĐD", "D"]))
    
    # Case changes
    if random.random() < params.get("upper_case_prob", 0):
        s = s.upper()
    elif random.random() < params.get("lower_case_prob", 0):
        s = s.lower()
    
    # Spacing noise
    if random.random() < params.get("spacing_noise_prob", 0):
        s = s.replace(", ", " ,  ")
    
    return s.strip()

# Extract cohort
print("Extracting cohort from SQLite...")
conn = sqlite3.connect("/content/ground_truth.db")
conn.row_factory = sqlite3.Row
cur = conn.cursor()
cur.execute("SELECT * FROM ground_truth")
all_rows = cur.fetchall()
conn.close()

print(f"[OK] Loaded {len(all_rows)} rows from ground_truth")

# Sample N=5000 with seed=3001
random.seed(3001)
cohort_rows = random.sample(all_rows, 5000)

# Apply noise
specimens = []
for row in cohort_rows:
    noisy = apply_noise(row['address'], "SUP-D2-1.0.0")  # High noise profile
    specimens.append({
        'gt_id': row['id'],
        'raw_address': noisy,
        'ref_address_v2': row['address'],
        'province_id': row['province_id'],
        'district_id': row['district_id'],
        'ward_id': row['ward_id']
    })

print(f"[OK] Created {len(specimens)} specimens with noise")
print(f"\nExample:")
print(f"  Original: {specimens[0]['ref_address_v2']}")
print(f"  Noisy:    {specimens[0]['raw_address']}")

In [ ]:
# === CELL 4: Run A1 (NER + TF-IDF + LLM) ===
import time
from tqdm import tqdm

print("="*60)
print("Running A1: NER + TF-IDF + LLM (Full)")
print("="*60)

results_a1 = []

for spec in tqdm(specimens, desc="A1 Progress"):
    start = time.time()
    
    try:
        # Step 1: NER
        ner_entities = ner.extract(spec['raw_address'])
        
        # Step 2: Retrieval
        candidates = retriever.retrieve_top_k(spec['raw_address'], top_k=5)
        
        # Step 3: LLM normalization
        pred_standardized, confidence, llm_latency = llm.normalize(
            query=spec['raw_address'],
            candidates=candidates
        )
        
        latency_ms = (time.time() - start) * 1000
        
        results_a1.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': pred_standardized,
            'latency_ms': latency_ms,
            'config': 'A1_FULL'
        })
    except Exception as e:
        print(f"\nError on specimen {spec['gt_id']}: {e}")
        results_a1.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': None,
            'latency_ms': None,
            'config': 'A1_FULL'
        })

# Calculate average latency safely
successful = [r for r in results_a1 if r['latency_ms'] is not None]
if successful:
    avg_latency = sum(r['latency_ms'] for r in successful) / len(successful)
    print(f"\n[OK] A1 completed: {len(successful)}/{len(results_a1)} successful predictions")
    print(f"  Avg latency: {avg_latency:.1f} ms")
else:
    print(f"\n[ERROR] A1 completed: 0/{len(results_a1)} successful predictions")
    print("  All predictions failed - check errors above")


In [ ]:
# === CELL 5: Run A2 (NER + TF-IDF, No LLM) ===
print("="*60)
print("Running A2: NER + TF-IDF (No LLM)")
print("="*60)

results_a2 = []

for spec in tqdm(specimens, desc="A2 Progress"):
    start = time.time()
    
    try:
        # NER + Retrieval, pick top-1
        ner_entities = ner.extract(spec['raw_address'])
        candidates = retriever.retrieve_top_k(spec['raw_address'], top_k=5)
        pred_standardized = candidates[0] if candidates else spec['raw_address']
        
        latency_ms = (time.time() - start) * 1000
        
        results_a2.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': pred_standardized,
            'latency_ms': latency_ms,
            'config': 'A2_NER_TFIDF'
        })
    except Exception as e:
        print(f"\nError: {e}")
        results_a2.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': None,
            'latency_ms': None,
            'config': 'A2_NER_TFIDF'
        })

successful = [r for r in results_a2 if r['latency_ms'] is not None]
if successful:
    avg_latency = sum(r['latency_ms'] for r in successful) / len(successful)
    print(f"\n[OK] A2 completed: {len(successful)}/{len(results_a2)} successful predictions")
    print(f"  Avg latency: {avg_latency:.1f} ms")
else:
    print(f"\n[ERROR] A2 completed: 0/{len(results_a2)} successful predictions")

for spec in tqdm(specimens, desc="A2 Progress"):
    start = time.time()
    
    try:
        # NER + Retrieval, pick top-1
        ner_entities = ner.extract(spec['raw_address'])
        candidates = retriever.retrieve_top_k(spec['raw_address'], top_k=5)
        pred_standardized = candidates[0] if candidates else spec['raw_address']
        
        latency_ms = (time.time() - start) * 1000
        
        results_a2.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': pred_standardized,
            'latency_ms': latency_ms,
            'config': 'A2_NER_MGTE'
        })
    except Exception as e:
        print(f"\nError: {e}")
        results_a2.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': None,
            'latency_ms': None,
            'config': 'A2_NER_MGTE'
        })

avg_latency = sum(r['latency_ms'] for r in results_a2 if r['latency_ms']) / len([r for r in results_a2 if r['latency_ms']])
print(f"\n[OK] A2 completed: {len(results_a2)} predictions")
print(f"  Avg latency: {avg_latency:.1f} ms")

In [ ]:
# === CELL 6: Run A3 (TF-IDF only) ===
print("="*60)
print("Running A3: TF-IDF only (Retrieval-only)")
print("="*60)

results_a3 = []

for spec in tqdm(specimens, desc="A3 Progress"):
    start = time.time()
    
    try:
        # Retrieval only
        candidates = retriever.retrieve_top_k(spec['raw_address'], top_k=5)
        pred_standardized = candidates[0] if candidates else spec['raw_address']
        
        latency_ms = (time.time() - start) * 1000
        
        results_a3.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': pred_standardized,
            'latency_ms': latency_ms,
            'config': 'A3_TFIDF_ONLY'
        })
    except Exception as e:
        print(f"\nError: {e}")
        results_a3.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': None,
            'latency_ms': None,
            'config': 'A3_TFIDF_ONLY'
        })

successful = [r for r in results_a3 if r['latency_ms'] is not None]
if successful:
    avg_latency = sum(r['latency_ms'] for r in successful) / len(successful)
    print(f"\n[OK] A3 completed: {len(successful)}/{len(results_a3)} successful predictions")
    print(f"  Avg latency: {avg_latency:.1f} ms")
else:
    print(f"\n[ERROR] A3 completed: 0/{len(results_a3)} successful predictions")
print("="*60)
print("Running A3: mGTE only (Retrieval-only)")
print("="*60)

results_a3 = []

for spec in tqdm(specimens, desc="A3 Progress"):
    start = time.time()
    
    try:
        # Retrieval only
        candidates = retriever.retrieve_top_k(spec['raw_address'], top_k=5)
        pred_standardized = candidates[0] if candidates else spec['raw_address']
        
        latency_ms = (time.time() - start) * 1000
        
        results_a3.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': pred_standardized,
            'latency_ms': latency_ms,
            'config': 'A3_MGTE_ONLY'
        })
    except Exception as e:
        print(f"\nError: {e}")
        results_a3.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': None,
            'latency_ms': None,
            'config': 'A3_MGTE_ONLY'
        })

avg_latency = sum(r['latency_ms'] for r in results_a3 if r['latency_ms']) / len([r for r in results_a3 if r['latency_ms']])
print(f"\n[OK] A3 completed: {len(results_a3)} predictions")
print(f"  Avg latency: {avg_latency:.1f} ms")

In [ ]:
# === CELL 7: Run A4 (NER + LLM, No Retrieval) ===
print("="*60)
print("Running A4: NER + LLM (No Retrieval)")
print("="*60)

results_a4 = []

for spec in tqdm(specimens, desc="A4 Progress"):
    start = time.time()
    
    try:
        # NER + LLM (no retrieval candidates)
        ner_entities = ner.extract(spec['raw_address'])
        pred_standardized, confidence, llm_latency = llm.normalize(
            query=spec['raw_address'],
            candidates=[]  # No retrieval
        )
        
        latency_ms = (time.time() - start) * 1000
        
        results_a4.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': pred_standardized,
            'latency_ms': latency_ms,
            'config': 'A4_NER_LLM'
        })
    except Exception as e:
        print(f"\nError: {e}")
        results_a4.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': None,
            'latency_ms': None,
            'config': 'A4_NER_LLM'
        })

successful = [r for r in results_a4 if r['latency_ms'] is not None]
if successful:
    avg_latency = sum(r['latency_ms'] for r in successful) / len(successful)
    print(f"\n[OK] A4 completed: {len(successful)}/{len(results_a4)} successful predictions")
    print(f"  Avg latency: {avg_latency:.1f} ms")
else:
    print(f"\n[ERROR] A4 completed: 0/{len(results_a4)} successful predictions")
print("="*60)
print("Running A4: NER + LLM (No Retrieval)")
print("="*60)

results_a4 = []

for spec in tqdm(specimens, desc="A4 Progress"):
    start = time.time()
    
    try:
        # NER + LLM (no retrieval candidates)
        ner_entities = ner.extract(spec['raw_address'])
        pred_standardized, confidence, llm_latency = llm.normalize(
            query=spec['raw_address'],
            candidates=[]  # No retrieval
        )
        
        latency_ms = (time.time() - start) * 1000
        
        results_a4.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': pred_standardized,
            'latency_ms': latency_ms,
            'config': 'A4_NER_LLM'
        })
    except Exception as e:
        print(f"\nError: {e}")
        results_a4.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': None,
            'latency_ms': None,
            'config': 'A4_NER_LLM'
        })

successful = [r for r in results_a4 if r['latency_ms'] is not None]
if successful:
    avg_latency = sum(r['latency_ms'] for r in successful) / len(successful)
    print(f"\n[OK] A4 completed: {len(successful)}/{len(results_a4)} successful predictions")
    print(f"  Avg latency: {avg_latency:.1f} ms")
else:
    print(f"\n[ERROR] A4 completed: 0/{len(results_a4)} successful predictions")


In [ ]:
# === CELL 8: Export Results ===
import pandas as pd
from google.colab import files

# Combine all results
all_results = results_a1 + results_a2 + results_a3 + results_a4

df = pd.DataFrame(all_results)
df.to_csv("ablation_n1000_results.csv", index=False)

print("="*60)
print("EXPORT SUMMARY")
print("="*60)
print(f"Total rows: {len(df)}")
print(f"\nBy config:")
print(df.groupby('config').agg({
    'latency_ms': ['count', 'mean', 'std']
}))

print(f"\n✓ Exported to ablation_n1000_results.csv")
print(f"\nDownloading...")
files.download("ablation_n1000_results.csv")